# Notebook 1: The b-value Stability Atlas

This notebook maps both the Gutenberg-Richter b-value and its temporal volatility across the globe using 25 years of USGS earthquake data. The b-value describes the relative proportion of small to large earthquakes in a region; deviations from the canonical b ≈ 1.0 and temporal instability in b can reveal tectonic stress changes, fluid injection effects, or volcanic unrest. We compute b-values on a 2°×2° global grid, measure their coefficient of variation (CV_b) over sliding time windows, examine the relationship between b-value and depth, and track regional b-value evolution for six tectonically distinct zones.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import (
    estimate_b_value, compute_bvalue_grid, compute_cv_b, rolling_b_value
)
from src.spatial import assign_cells
from src.plotting import (
    setup_style, save_figure, plot_global_map, plot_bvalue_volatility_map
)
from src.external_data import (
    load_gcmt_catalog, load_gsrm_strain, resample_strain_to_grid,
    load_heat_flow, resample_heat_flow_to_grid,
    load_pb2002_boundaries, classify_grid_tectonic_settings
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")
print(f"Date range: {df['time'].min()} \u2192 {df['time'].max()}")
print(f"Magnitude range: {df['mag'].min():.1f} \u2192 {df['mag'].max():.1f}")

Loaded 681,450 events
Date range: 2000-01-01 01:19:26.990000+00:00 → 2025-12-31 23:52:11.327000+00:00
Magnitude range: 0.7 → 9.1


## 1.1 Global b-value Map

In [3]:
# Compute b-values on 2°×2° grid
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)
print(f"Cells with valid b-values: {len(bvalue_grid)}")
print(f"b-value range: {bvalue_grid['b_value'].min():.2f} \u2192 {bvalue_grid['b_value'].max():.2f}")
print(f"Median b: {bvalue_grid['b_value'].median():.2f}")

Cells with valid b-values: 709
b-value range: 0.40 → 2.18
Median b: 1.09


In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_global_map(
    bvalue_grid["cell_lat"], bvalue_grid["cell_lon"], bvalue_grid["b_value"],
    cmap="RdYlBu_r", vmin=0.6, vmax=1.4,
    label="b-value (MLE)", title="Global Gutenberg-Richter b-value (2000\u20132025)",
    ax=ax
)
save_figure(fig, "01_global_bvalue_map")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/3661931277.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.2 Temporal Volatility (CV_b)

In [5]:
cv_grid = compute_cv_b(df, cell_size=2.0, window_years=3, stride_years=1,
                        min_events_per_window=50, min_events_total=200)
print(f"Cells with CV_b: {len(cv_grid)}")
print(f"CV_b range: {cv_grid['cv_b'].min():.3f} \u2192 {cv_grid['cv_b'].max():.3f}")
print(f"Median CV_b: {cv_grid['cv_b'].median():.3f}")

Cells with CV_b: 303
CV_b range: 0.000 → 0.594
Median CV_b: 0.108


In [6]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_bvalue_volatility_map(cv_grid["cell_lat"], cv_grid["cell_lon"], cv_grid["cv_b"], ax=ax)
save_figure(fig, "01_global_cv_b_map")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/55170868.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3 b-value vs. Depth

In [7]:
# Merge depth info with b-value grid
df_cells = assign_cells(df, cell_size=2.0)
depth_by_cell = df_cells.groupby(["cell_lat", "cell_lon"])["depth"].median().reset_index()
depth_by_cell.columns = ["cell_lat", "cell_lon", "median_depth"]

bv_depth = bvalue_grid.merge(depth_by_cell, on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(bv_depth["median_depth"], bv_depth["b_value"],
                c=bv_depth["n_events"], cmap="viridis", s=15, alpha=0.6,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax, label="Number of events")
ax.set_xlabel("Median Depth (km)")
ax.set_ylabel("b-value")
ax.set_title("b-value vs. Depth (2\u00b0\u00d72\u00b0 cells)")
ax.set_xlim(0, 700)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
save_figure(fig, "01_bvalue_vs_depth")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/3495311201.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3a b-value by Faulting Style (Global CMT)

Using the Global CMT catalog (40,000+ moment tensor solutions), we classify each event as thrust, normal, strike-slip, or oblique and examine how b-value varies systematically with faulting regime. Theory predicts b ≈ 0.7 for thrust, b ≈ 1.0 for strike-slip, and b ≈ 1.2 for normal faulting (Schorlemmer et al. 2005).

In [8]:
# Load Global CMT catalog
gcmt = load_gcmt_catalog(min_year=2000)
print(f"GCMT events (2000+): {len(gcmt):,}")
print(f"Faulting distribution:\n{gcmt['faulting'].value_counts()}")

# b-value by faulting style
faulting_colors = {"thrust": "#E63946", "normal": "#2A9D8F", "strike-slip": "#457B9D", "oblique": "#AAAAAA"}
faulting_order = ["thrust", "strike-slip", "normal", "oblique"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: b-value by faulting type
b_by_fault = {}
for ft in faulting_order:
    mags = gcmt[gcmt["faulting"] == ft]["Mw"].values
    mc = estimate_mc(mags)
    above = mags[mags >= mc]
    if len(above) >= 50:
        result = estimate_b_value(above, mc)
        b_val = result[0] if isinstance(result, tuple) else result
        b_by_fault[ft] = {"b": b_val, "mc": mc, "n": len(above)}

ax = axes[0]
bars = ax.bar(list(b_by_fault.keys()),
              [v["b"] for v in b_by_fault.values()],
              color=[faulting_colors[ft] for ft in b_by_fault.keys()],
              alpha=0.8)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("b-value (MLE)")
ax.set_title("b-value by Faulting Style (GCMT)")
for i, (ft, v) in enumerate(b_by_fault.items()):
    ax.text(i, v["b"] + 0.02, f"b={v['b']:.2f}\nn={v['n']:,}", ha="center", fontsize=9)

# Panel 2: Magnitude-frequency by faulting type
ax = axes[1]
for ft in faulting_order:
    mags = gcmt[gcmt["faulting"] == ft]["Mw"].values
    bins = np.arange(4.5, 9.0, 0.1)
    counts, edges = np.histogram(mags, bins=bins)
    cum = np.cumsum(counts[::-1])[::-1]
    mask = cum > 0
    ax.semilogy(edges[:-1][mask], cum[mask], "-", color=faulting_colors[ft],
                label=ft, linewidth=1.5)
ax.set_xlabel("Magnitude (Mw)")
ax.set_ylabel("Cumulative count (≥M)")
ax.set_title("Frequency-Magnitude by Faulting Style")
ax.legend()

fig.tight_layout()
save_figure(fig, "01_bvalue_by_faulting")
plt.show()

for ft, v in b_by_fault.items():
    print(f"  {ft:12s}: b={v['b']:.3f}, Mc={v['mc']:.1f}, n={v['n']:,}")

GCMT events (2000+): 40,738
Faulting distribution:
faulting
strike-slip    15270
thrust         11327
oblique         7262
normal          6879
Name: count, dtype: int64
  thrust      : b=0.860, Mc=5.2, n=5,838
  strike-slip : b=0.924, Mc=5.1, n=9,096
  normal      : b=1.017, Mc=5.1, n=3,297
  oblique     : b=0.942, Mc=5.1, n=3,953


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/952272020.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3b b-value vs. Geodetic Strain Rate (GSRM v2.1)

We test whether b-value correlates with geodetic strain rate by joining the GSRM v2.1 strain rate model (0.1° resolution, resampled to 2°) with our b-value grid. Higher strain rates indicate faster tectonic loading; if b-value reflects differential stress, we expect an inverse relationship.

In [9]:
# Load and resample GSRM strain rates to 2° grid
strain_raw = load_gsrm_strain()
strain_grid = resample_strain_to_grid(strain_raw, grid_size=2.0)
print(f"GSRM strain rate grid: {len(strain_grid)} cells at 2° resolution")

# Join with b-value grid
bv_strain = bvalue_grid.merge(
    strain_grid, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
bv_strain = bv_strain[bv_strain["mean_second_invariant"] > 0]  # exclude zero-strain cells
print(f"Matched cells (b-value + strain): {len(bv_strain)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: b-value vs strain rate
ax = axes[0]
sc = ax.scatter(bv_strain["mean_second_invariant"], bv_strain["b_value"],
                c=bv_strain["n_events"], cmap="viridis", s=15, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
ax.set_xscale("log")
ax.set_xlabel("Geodetic Strain Rate (nanostr/yr)")
ax.set_ylabel("b-value")
ax.set_title("b-value vs. Geodetic Strain Rate")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
plt.colorbar(sc, ax=ax, label="Number of events")

# Correlation in log space
from scipy.stats import spearmanr
log_strain = np.log10(bv_strain["mean_second_invariant"])
rho, p = spearmanr(log_strain, bv_strain["b_value"])
ax.text(0.05, 0.95, f"Spearman ρ = {rho:.3f}\np = {p:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Panel 2: CV_b vs strain rate
cv_strain = cv_grid.merge(
    strain_grid, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
cv_strain = cv_strain[cv_strain["mean_second_invariant"] > 0]

ax = axes[1]
ax.scatter(cv_strain["mean_second_invariant"], cv_strain["cv_b"],
           c="#2A9D8F", s=15, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Geodetic Strain Rate (nanostr/yr)")
ax.set_ylabel("CV_b (temporal volatility)")
ax.set_title("b-value Volatility vs. Strain Rate")

rho2, p2 = spearmanr(np.log10(cv_strain["mean_second_invariant"]), cv_strain["cv_b"])
ax.text(0.05, 0.95, f"Spearman ρ = {rho2:.3f}\np = {p2:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.tight_layout()
save_figure(fig, "01_bvalue_vs_strain_rate")
plt.show()

GSRM strain rate grid: 3040 cells at 2° resolution
Matched cells (b-value + strain): 665


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/865903034.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3c b-value vs. Heat Flow (IHFC Global Heat Flow Database)

We test whether b-value correlates with crustal thermal regime using the IHFC Global Heat Flow Database (2024 release, ~91,000 measurements). Higher heat flow implies warmer, weaker crust with lower differential stress, which theory predicts should produce higher b-values. We resample point measurements to a 2°×2° grid (median per cell) and join with our b-value grid.

The 2024 GHFDB release includes coded quality scores per Fuchs et al. (2023): a **U-score** (uncertainty quality, 1–4) and **M-score** (methodology quality, 1–4). We compare the correlation using all measurements versus only high-quality ones (U1–U2: excellent/good uncertainty, <15% COV).

In [10]:
# Load ALL heat flow data (unfiltered)
hf_raw = load_heat_flow()
hf_raw = hf_raw[(hf_raw["q_mw_m2"] > 0) & (hf_raw["q_mw_m2"] < 1000)].copy()
print(f"Heat flow measurements (all, after outlier filter): {len(hf_raw):,}")

# Quality score breakdown
if "u_score" in hf_raw.columns:
    scored = hf_raw["u_score"].notna().sum()
    print(f"  With U-score: {scored:,} ({100*scored/len(hf_raw):.0f}%)")
    print(f"  U-score distribution: {hf_raw['u_score'].value_counts().sort_index().to_dict()}")

# Load quality-filtered subset (U1+U2 = excellent+good uncertainty)
hf_quality = load_heat_flow(min_quality="2")
hf_quality = hf_quality[(hf_quality["q_mw_m2"] > 0) & (hf_quality["q_mw_m2"] < 1000)].copy()
print(f"\nHigh-quality measurements (U1–U2, <15% COV): {len(hf_quality):,}")
print(f"  Median heat flow: {hf_quality['q_mw_m2'].median():.1f} mW/m²")

# Grid both datasets
hf_grid_all = resample_heat_flow_to_grid(hf_raw, grid_size=2.0)
hf_grid_quality = resample_heat_flow_to_grid(hf_quality, grid_size=2.0)
print(f"\nGrid cells (all): {len(hf_grid_all):,}")
print(f"Grid cells (quality-filtered): {len(hf_grid_quality):,}")

# Join both with b-value grid
bv_hf_all = bvalue_grid.merge(
    hf_grid_all, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
bv_hf_all = bv_hf_all[bv_hf_all["n_measurements"] >= 3].copy()

bv_hf_qual = bvalue_grid.merge(
    hf_grid_quality, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
bv_hf_qual = bv_hf_qual[bv_hf_qual["n_measurements"] >= 3].copy()

print(f"\nMatched cells (all, ≥3 measurements): {len(bv_hf_all)}")
print(f"Matched cells (quality, ≥3 measurements): {len(bv_hf_qual)}")

# Also join CV_b with all heat flow
cv_hf = cv_grid.merge(
    hf_grid_all, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
cv_hf = cv_hf[cv_hf["n_measurements"] >= 3].copy()

from scipy.stats import spearmanr

# --- Compute correlations ---
rho_all, p_all = spearmanr(bv_hf_all["median_q"], bv_hf_all["b_value"])
rho_qual, p_qual = spearmanr(bv_hf_qual["median_q"], bv_hf_qual["b_value"])
rho_cv, p_cv = spearmanr(cv_hf["median_q"], cv_hf["cv_b"])

# --- 4-panel figure ---
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Panel 1: b-value vs heat flow (ALL measurements)
ax = axes[0, 0]
sc = ax.scatter(bv_hf_all["median_q"], bv_hf_all["b_value"],
                c=bv_hf_all["n_events"], cmap="viridis", s=18, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
ax.set_xlabel("Median Heat Flow (mW/m²)")
ax.set_ylabel("b-value")
ax.set_title("All Measurements (n_cells={})".format(len(bv_hf_all)))
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
plt.colorbar(sc, ax=ax, label="Earthquake count", shrink=0.8)
ax.text(0.05, 0.95, f"Spearman ρ = {rho_all:.3f}\np = {p_all:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Panel 2: b-value vs heat flow (QUALITY-FILTERED U1+U2)
ax = axes[0, 1]
sc = ax.scatter(bv_hf_qual["median_q"], bv_hf_qual["b_value"],
                c=bv_hf_qual["n_events"], cmap="viridis", s=18, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
ax.set_xlabel("Median Heat Flow (mW/m²)")
ax.set_ylabel("b-value")
ax.set_title("Quality-Filtered U1–U2 (n_cells={})".format(len(bv_hf_qual)))
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
plt.colorbar(sc, ax=ax, label="Earthquake count", shrink=0.8)
ax.text(0.05, 0.95, f"Spearman ρ = {rho_qual:.3f}\np = {p_qual:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="#90EE90", alpha=0.5))

# Panel 3: CV_b vs heat flow
ax = axes[1, 0]
ax.scatter(cv_hf["median_q"], cv_hf["cv_b"], c="#E63946", s=18, alpha=0.5)
ax.set_xlabel("Median Heat Flow (mW/m²)")
ax.set_ylabel("CV_b (temporal volatility)")
ax.set_title("b-value Volatility vs. Heat Flow")
ax.text(0.05, 0.95, f"Spearman ρ = {rho_cv:.3f}\np = {p_cv:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Panel 4: Heat flow binned analysis (quality-filtered)
ax = axes[1, 1]
bin_edges = [0, 40, 60, 80, 100, 150, 1000]
bin_labels = ["<40", "40–60", "60–80", "80–100", "100–150", ">150"]

# Use quality-filtered for binned analysis
bv_hf_qual["hf_bin"] = pd.cut(bv_hf_qual["median_q"], bins=bin_edges, labels=bin_labels)
binned = bv_hf_qual.groupby("hf_bin", observed=True).agg(
    median_b=("b_value", "median"),
    q25=("b_value", lambda x: x.quantile(0.25)),
    q75=("b_value", lambda x: x.quantile(0.75)),
    n=("b_value", "count"),
).reset_index()
binned = binned[binned["n"] >= 2]  # need at least 2 cells per bin

colors = ["#457B9D", "#457B9D", "#2A9D8F", "#2A9D8F", "#F4A261", "#E63946"]
bars = ax.bar(range(len(binned)), binned["median_b"], color=colors[:len(binned)], alpha=0.8)
ax.errorbar(range(len(binned)), binned["median_b"],
            yerr=[binned["median_b"] - binned["q25"], binned["q75"] - binned["median_b"]],
            fmt="none", color="black", capsize=4)
ax.set_xticks(range(len(binned)))
ax.set_xticklabels(binned["hf_bin"], rotation=30, ha="right")
ax.set_xlabel("Heat Flow Bin (mW/m²)")
ax.set_ylabel("Median b-value")
ax.set_title("b-value by Heat Flow Regime (Quality-Filtered)")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
for i, row in binned.iterrows():
    ax.text(i, row["q75"] + 0.03, f"n={row['n']}", ha="center", fontsize=8)

fig.suptitle("b-value and Heat Flow: Testing the Thermal Weakness Hypothesis\n"
             "(Fuchs et al. 2023 quality scoring applied)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "01_bvalue_vs_heat_flow")
plt.show()

print(f"\nCorrelation comparison:")
print(f"  All measurements:      ρ = {rho_all:.3f}, p = {p_all:.2e} (n={len(bv_hf_all)} cells)")
print(f"  Quality-filtered U1–U2: ρ = {rho_qual:.3f}, p = {p_qual:.2e} (n={len(bv_hf_qual)} cells)")
print(f"  CV_b vs heat flow:      ρ = {rho_cv:.3f}, p = {p_cv:.2e}")
print(f"\nBinned b-values (quality-filtered):")
for _, row in binned.iterrows():
    print(f"  {row['hf_bin']:>8s}: median b = {row['median_b']:.3f} (n={row['n']})")

Heat flow measurements (all, after outlier filter): 89,491
  With U-score: 14,947 (17%)
  U-score distribution: {1.0: 2186, 2.0: 7297, 3.0: 3416, 4.0: 2048}

High-quality measurements (U1–U2, <15% COV): 9,483
  Median heat flow: 69.3 mW/m²

Grid cells (all): 5,469
Grid cells (quality-filtered): 1,470

Matched cells (all, ≥3 measurements): 383
Matched cells (quality, ≥3 measurements): 91

Correlation comparison:
  All measurements:      ρ = -0.172, p = 7.05e-04 (n=383 cells)
  Quality-filtered U1–U2: ρ = -0.207, p = 4.91e-02 (n=91 cells)
  CV_b vs heat flow:      ρ = -0.126, p = 9.69e-02

Binned b-values (quality-filtered):
       <40: median b = 0.959 (n=16)
     40–60: median b = 1.114 (n=21)
     60–80: median b = 1.145 (n=20)
    80–100: median b = 0.925 (n=11)
   100–150: median b = 0.833 (n=12)
      >150: median b = 0.856 (n=11)


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/1389385522.py:126: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3d b-value by Tectonic Setting (PB2002 Plate Boundary Model)

Using the PB2002 plate boundary model (Bird, 2003), we classify each 2°×2° grid cell by its dominant tectonic setting: subduction, convergent, transform, spreading, rift, or intraplate. This provides the categorical variable needed to test whether b-value variations are systematically explained by plate boundary type.

In [11]:
# Classify each b-value grid cell by tectonic setting
boundaries = load_pb2002_boundaries()
print(f"PB2002 boundary segments: {len(boundaries):,}")
print(f"Boundary types: {boundaries['boundary_type'].value_counts().to_dict()}")

settings = classify_grid_tectonic_settings(
    bvalue_grid["cell_lat"].values, bvalue_grid["cell_lon"].values,
    boundaries, radius_deg=2.0
)
bvalue_grid["tectonic_setting"] = settings

setting_counts = bvalue_grid["tectonic_setting"].value_counts()
print(f"\nGrid cells by tectonic setting:\n{setting_counts}")

# Also classify CV_b grid
cv_settings = classify_grid_tectonic_settings(
    cv_grid["cell_lat"].values, cv_grid["cell_lon"].values,
    boundaries, radius_deg=2.0
)
cv_grid["tectonic_setting"] = cv_settings

# --- Visualization ---
setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: b-value by tectonic setting (box plot)
ax = axes[0]
plot_data = []
plot_labels = []
plot_colors = []
for s in setting_order:
    vals = bvalue_grid[bvalue_grid["tectonic_setting"] == s]["b_value"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        plot_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("b-value")
ax.set_title("b-value by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 2: CV_b by tectonic setting
ax = axes[1]
plot_data = []
plot_labels = []
plot_colors = []
for s in setting_order:
    vals = cv_grid[cv_grid["tectonic_setting"] == s]["cv_b"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        plot_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("CV_b (temporal volatility)")
ax.set_title("b-value Volatility by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 3: Global map colored by tectonic setting
ax = axes[2]
for s in setting_order:
    mask = bvalue_grid["tectonic_setting"] == s
    if mask.sum() > 0:
        ax.scatter(bvalue_grid.loc[mask, "cell_lon"],
                   bvalue_grid.loc[mask, "cell_lat"],
                   c=setting_colors[s], s=8, alpha=0.6, label=s)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Grid Cell Tectonic Classification")
ax.legend(fontsize=8, loc="lower left", ncol=2)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)

fig.suptitle("b-value and Temporal Volatility by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "01_bvalue_by_tectonic_setting")
plt.show()

# Summary statistics
from scipy.stats import kruskal
print("\nMedian b-value by tectonic setting:")
for s in setting_order:
    vals = bvalue_grid[bvalue_grid["tectonic_setting"] == s]["b_value"]
    if len(vals) >= 5:
        print(f"  {s:15s}: median={vals.median():.3f}, mean={vals.mean():.3f}, "
              f"std={vals.std():.3f}, n={len(vals)}")

# Kruskal-Wallis test (non-parametric ANOVA)
groups = [bvalue_grid[bvalue_grid["tectonic_setting"] == s]["b_value"].values
          for s in setting_order
          if len(bvalue_grid[bvalue_grid["tectonic_setting"] == s]) >= 5]
H, p = kruskal(*groups)
print(f"\nKruskal-Wallis H = {H:.1f}, p = {p:.2e}")

print("\nMedian CV_b by tectonic setting:")
for s in setting_order:
    vals = cv_grid[cv_grid["tectonic_setting"] == s]["cv_b"]
    if len(vals) >= 5:
        print(f"  {s:15s}: median={vals.median():.3f}, n={len(vals)}")

PB2002 boundary segments: 5,819
Boundary types: {'OSR': 1875, 'OTF': 1145, 'SUB': 1127, 'CRB': 474, 'CTF': 456, 'CCB': 401, 'OCB': 341}

Grid cells by tectonic setting:
tectonic_setting
subduction    289
intraplate    158
transform      90
spreading      65
convergent     60
rift           47
Name: count, dtype: int64


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/3539796156.py:43: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/3539796156.py:65: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,



Median b-value by tectonic setting:
  subduction     : median=1.110, mean=1.095, std=0.280, n=289
  convergent     : median=1.067, mean=1.059, std=0.274, n=60
  transform      : median=1.045, mean=1.043, std=0.266, n=90
  spreading      : median=1.210, mean=1.210, std=0.333, n=65
  rift           : median=1.178, mean=1.218, std=0.340, n=47
  intraplate     : median=0.954, mean=0.998, std=0.263, n=158

Kruskal-Wallis H = 39.3, p = 2.11e-07

Median CV_b by tectonic setting:
  subduction     : median=0.116, n=151
  convergent     : median=0.099, n=18
  transform      : median=0.079, n=33
  spreading      : median=0.111, n=14
  rift           : median=0.151, n=23
  intraplate     : median=0.080, n=64


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/3539796156.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.4 Regional b-value Time Series

In [12]:
regions = {
    "Japan Trench": {"lat": (30, 45), "lon": (135, 150)},
    "San Andreas": {"lat": (32, 37), "lon": (-122, -115)},
    "Oklahoma": {"lat": (33.5, 37.5), "lon": (-100, -94.5)},
    "Sumatra": {"lat": (-5, 10), "lon": (92, 106)},
    "Iceland": {"lat": (63, 67), "lon": (-25, -13)},
    "Yellowstone": {"lat": (43, 46), "lon": (-112, -109)},
}

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
axes_flat = axes.flatten()

for i, (name, bounds) in enumerate(regions.items()):
    ax = axes_flat[i]
    mask = (
        (df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
        (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1])
    )
    region_df = df[mask].copy()

    if len(region_df) < 200:
        ax.set_title(f"{name} (insufficient data)")
        continue

    mc = estimate_mc(region_df["mag"].values)
    rolling = rolling_b_value(
        region_df["time"].values, region_df["mag"].values,
        mc=mc, window_size=200, step=50
    )

    if len(rolling) > 0:
        ax.plot(
            pd.to_datetime(rolling["center_time"]), rolling["b_value"],
            color="#2A9D8F", linewidth=1
        )
        ax.fill_between(
            pd.to_datetime(rolling["center_time"]),
            rolling["b_value"] - rolling["b_std"],
            rolling["b_value"] + rolling["b_std"],
            alpha=0.2, color="#2A9D8F"
        )
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)

    ax.set_title(f"{name} (Mc={mc:.1f}, n={len(region_df):,})")
    ax.set_ylabel("b-value")

axes_flat[-2].set_xlabel("Date")
axes_flat[-1].set_xlabel("Date")
fig.suptitle(
    "Regional b-value Time Series (rolling 200-event windows)",
    fontsize=14, fontweight="bold"
)
fig.tight_layout()
save_figure(fig, "01_regional_bvalue_timeseries")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_67712/157847126.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook constructed a global atlas of Gutenberg-Richter b-values and their temporal stability. Key observations:

- The global median b-value clusters near the canonical value of 1.0, but substantial regional variation exists across tectonic settings.
- Subduction zones and volcanic regions tend to show elevated b-values, while stable continental interiors trend lower.
- The coefficient of variation (CV_b) highlights regions where b-values are temporally unstable, which may indicate transient stress perturbations, fluid injection, or evolving fault networks.
- The b-value vs. depth analysis reveals systematic trends consistent with increasing differential stress at depth.
- Regional time series show that some zones (e.g., Oklahoma) exhibit pronounced b-value shifts coinciding with known changes in industrial activity, while purely tectonic regions maintain more stable b-values over time.

These spatial and temporal b-value patterns form the baseline for identifying anomalous seismicity signatures in subsequent notebooks.